# Retraining v2 — Safety-Focused (Near-Miss Penalty + Stronger Collision Penalty)

**Fully isolated**: imports `core_v2.py` (this folder), reads `outputs/dueling_per_phase1` (read-only),
writes everything new to `outputs_v2/`.

Changes vs. original `core.py`:
1. `PEDESTRIAN_COLLISION_REWARD`: -100 → -300
2. **NEW** near-miss penalty: up to -15 when a pedestrian ends within 20% of sensor range,
   even without a collision — teaches early avoidance.

Strategy: load the **Phase 1 checkpoint** (before pedestrians ever appear — navigation skills
are solid, no pedestrian-related habits yet), then retrain Phase 2 → 3 → 4 **from scratch**
with the new reward, using the full original episode counts (6000 + 8000 + 8000 = 22,000).

In [1]:
import os, sys, time, json
import numpy as np
import importlib.util

ORIGINAL_DIR = os.path.join('..', 'outputs')   # read-only
V2_DIR       = '.'                              # this folder (outputs_v2/)

spec = importlib.util.spec_from_file_location("core_v2", os.path.join(V2_DIR, "core_v2.py"))
core_v2 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(core_v2)

BlindNavigatorEnv     = core_v2.BlindNavigatorEnv
DuelingPERAgent       = core_v2.DuelingPERAgent
run_episode           = core_v2.run_episode
evaluate              = core_v2.evaluate
get_epsilon           = core_v2.get_epsilon
PHASE_CONFIG          = core_v2.PHASE_CONFIG
PHASE_EPISODES        = core_v2.PHASE_EPISODES
PHASE_EPSILON_RESTART = core_v2.PHASE_EPSILON_RESTART

print(f'PEDESTRIAN_COLLISION_REWARD (v2) = {core_v2.PEDESTRIAN_COLLISION_REWARD}')
print(f'NEAR_MISS_PENALTY_MAX (v2)       = {core_v2.NEAR_MISS_PENALTY_MAX}')
print(f'NEAR_MISS_THRESHOLD (v2)          = {core_v2.NEAR_MISS_THRESHOLD}')
print(f'Reading checkpoints from : {ORIGINAL_DIR}/  (read-only)')
print(f'Writing all results to   : {V2_DIR}/')

PEDESTRIAN_COLLISION_REWARD (v2) = -300.0
NEAR_MISS_PENALTY_MAX (v2)       = -15.0
NEAR_MISS_THRESHOLD (v2)          = 0.2
Reading checkpoints from : ..\outputs/  (read-only)
Writing all results to   : ./


## Load Phase 1 checkpoint (pre-pedestrian navigation skills)

In [2]:
agent = DuelingPERAgent()
agent.load(os.path.join(ORIGINAL_DIR, 'dueling_per_phase1'))
print('✓ Loaded outputs/dueling_per_phase1 (read-only)')
print('  This checkpoint has solid navigation + static-hazard avoidance,')
print('  but has never seen a pedestrian.')

✓ Loaded outputs/dueling_per_phase1 (read-only)
  This checkpoint has solid navigation + static-hazard avoidance,
  but has never seen a pedestrian.


## Retrain Phase 2 → 3 → 4 with the new reward (full episode counts)

In [3]:
logs = []
log_path = os.path.join(V2_DIR, 'dueling_per_v2_curriculum_logs.json')

for phase in [2, 3, 4]:
    env = BlindNavigatorEnv(phase=phase)
    n_episodes = PHASE_EPISODES[phase]

    print(f"\nPhase {phase} — {PHASE_CONFIG[phase]['description']}")
    print(f"Episodes: {n_episodes} | ε₀: {PHASE_EPSILON_RESTART[phase]:.2f} | Maps: {PHASE_CONFIG[phase]['map_ids']}")

    for ep in range(n_episodes):
        eps = get_epsilon(ep, phase)
        seed = 500000 + len(logs)
        r, s, su, go, co = run_episode(env, agent, eps, seed)

        logs.append({
            'phase': phase, 'episode': ep, 'global_episode': len(logs),
            'total_reward': round(float(r), 3), 'steps': s,
            'success': bool(su), 'game_over': bool(go), 'collisions': int(co),
            'epsilon': round(float(eps), 5), 'seed': seed,
            'algorithm': agent.name, 'map_id': env.map_id,
        })

        if (ep + 1) % 200 == 0 or ep == n_episodes - 1:
            recent = logs[-200:]
            avg_r = np.mean([l['total_reward'] for l in recent])
            sr    = np.mean([l['success']      for l in recent])
            coll  = np.mean([l['collisions'] > 0 for l in recent])
            print(f"  [{ep+1:>5}/{n_episodes}] SR={sr:.1%} | AvgR={avg_r:7.1f} | "
                  f"CollEpRate={coll:.1%} | ε={eps:.3f}")
            with open(log_path, 'w') as f:
                json.dump(logs, f)

    # phase checkpoint
    agent.save(os.path.join(V2_DIR, f'dueling_per_v2_phase{phase}'))
    print(f'  ✓ Phase {phase} checkpoint saved')

agent.save(os.path.join(V2_DIR, 'dueling_per_v2_final'))
with open(log_path, 'w') as f:
    json.dump(logs, f)
print(f'\n✓ Retraining complete — {len(logs)} episodes')


Phase 2 — Moving pedestrians — reactive avoidance
Episodes: 6000 | ε₀: 0.60 | Maps: [0, 1, 2, 3, 4]
  [  200/6000] SR=3.5% | AvgR= -264.0 | CollEpRate=24.5% | ε=0.564
  [  400/6000] SR=11.0% | AvgR= -171.9 | CollEpRate=17.5% | ε=0.527
  [  600/6000] SR=35.5% | AvgR=  -58.1 | CollEpRate=20.0% | ε=0.490
  [  800/6000] SR=69.5% | AvgR=  188.4 | CollEpRate=18.0% | ε=0.454
  [ 1000/6000] SR=79.5% | AvgR=  263.1 | CollEpRate=16.0% | ε=0.417
  [ 1200/6000] SR=85.0% | AvgR=  350.4 | CollEpRate=15.0% | ε=0.380
  [ 1400/6000] SR=86.5% | AvgR=  373.0 | CollEpRate=14.0% | ε=0.344
  [ 1600/6000] SR=93.5% | AvgR=  436.3 | CollEpRate=9.5% | ε=0.307
  [ 1800/6000] SR=95.5% | AvgR=  485.6 | CollEpRate=7.0% | ε=0.270
  [ 2000/6000] SR=99.0% | AvgR=  491.7 | CollEpRate=9.5% | ε=0.234
  [ 2200/6000] SR=97.5% | AvgR=  496.6 | CollEpRate=5.5% | ε=0.197
  [ 2400/6000] SR=98.5% | AvgR=  523.9 | CollEpRate=3.5% | ε=0.160
  [ 2600/6000] SR=99.5% | AvgR=  518.0 | CollEpRate=5.0% | ε=0.124
  [ 2800/6000] SR=99.5

## Evaluate v2 (300 episodes, Phase 4 — hardest, all maps)

In [19]:
ev_v2 = evaluate(agent, n_episodes=300, output_dir=V2_DIR, phase=4)
print(f"  SR={ev_v2['success_rate']:.1%} | "
      f"R={ev_v2['mean_reward']:.1f} | "
      f"Collisions={ev_v2['collision_rate']:.1%} | "
      f"GameOver={ev_v2['game_over_rate']:.1%}")
ev_v2

  SR=100.0% | R=526.3 | Collisions=7.7% | GameOver=0.0%


{'algorithm': 'Dueling-PER',
 'n_episodes': 300,
 'success_rate': 1.0,
 'mean_reward': 526.299,
 'std_reward': 116.78,
 'mean_steps': 41.55,
 'collision_rate': 0.0767,
 'game_over_rate': 0.0,
 'per_map': {'map_2': {'success_rate': 1.0,
   'mean_reward': 555.35,
   'n_episodes': 56},
  'map_5': {'success_rate': 1.0, 'mean_reward': 516.4, 'n_episodes': 50},
  'map_3': {'success_rate': 1.0, 'mean_reward': 533.26, 'n_episodes': 60},
  'map_1': {'success_rate': 1.0, 'mean_reward': 486.03, 'n_episodes': 47},
  'map_4': {'success_rate': 1.0, 'mean_reward': 500.72, 'n_episodes': 44},
  'map_0': {'success_rate': 1.0, 'mean_reward': 560.46, 'n_episodes': 43}}}

## v1 vs v2 comparison

In [20]:
import pandas as pd

v1 = {'success_rate': 1.0, 'mean_reward': 545.413, 'collision_rate': 0.2300, 'game_over_rate': 0.0}

comparison_df = pd.DataFrame({
    'v1 (original)': v1,
    'v2 (near-miss penalty, full retrain)': {
        'success_rate': ev_v2['success_rate'],
        'mean_reward': ev_v2['mean_reward'],
        'collision_rate': ev_v2['collision_rate'],
        'game_over_rate': ev_v2['game_over_rate'],
    }
})
comparison_df['change'] = comparison_df['v2 (near-miss penalty, full retrain)'] - comparison_df['v1 (original)']
comparison_df

,v1 (original),"v2 (near-miss penalty, full retrain)",change
success_rate,1.000,1.0000,0.0000
mean_reward,545.413,526.2990,-19.1140
collision_rate,0.230,0.0767,-0.1533
game_over_rate,0.000,0.0000,0.0000


In [21]:
per_map_df = pd.DataFrame(ev_v2['per_map']).T
per_map_df

,success_rate,mean_reward,n_episodes
map_2,1.0,555.35,56.0
map_5,1.0,516.40,50.0
map_3,1.0,533.26,60.0
map_1,1.0,486.03,47.0
map_4,1.0,500.72,44.0
map_0,1.0,560.46,43.0


In [22]:
with open(os.path.join(V2_DIR, 'v2_summary.json'), 'w') as f:
    json.dump({'v1': v1, 'v2': ev_v2}, f, indent=2, default=str)

print(f'✓ All results saved to {V2_DIR}/')
print(f'✓ Original ../outputs/ and core.py completely untouched')

✓ All results saved to ./
✓ Original ../outputs/ and core.py completely untouched
